# Note 08 Deep Dive: Dot Product, Projections & Least Squares

Companion notebook for [`08-Dot-Product-Projections-Least-Squares.md`](../08-Dot-Product-Projections-Least-Squares.md). Every number quoted in the note is produced here.

| Part | Topic | Note section |
|---|---|---|
| A | Projections onto the axes (board example) | §2 |
| B | Dot product and its properties | §3 |
| C | Angle, Cauchy–Schwarz, cosine similarity | §4 |
| D | Projection of a vector onto a vector | §5 |
| E | The projection matrix vvᵀ/vᵀv: symmetric, idempotent, rank 1, eigenvalues | §6 |
| F | Projection onto a subspace: A(AᵀA)⁻¹Aᵀ | §7 |
| G | Least squares by hand vs `lstsq` vs scikit-learn; residuals; hat matrix | §8 |
| H | Pseudo-inverse when columns are dependent | §9 |
| I | Gram–Schmidt teaser | §10 |
| J | Case studies | §12 |
| K | Practice-problem checks | §17 |

Figures for the note are produced by [`figures_08.py`](figures_08.py).

In [1]:
import numpy as np
import sympy as sp
from sklearn.linear_model import LinearRegression
np.set_printoptions(precision=4, suppress=True)

def proj_vec(u, v):
    u, v = np.asarray(u, float), np.asarray(v, float)
    return (u @ v) / (v @ v) * v

def proj_matrix(v):
    v = sp.Matrix(v)
    return (v * v.T) / (v.T * v)[0]

def angle_deg(u, v):
    u, v = np.asarray(u, float), np.asarray(v, float)
    return np.degrees(np.arccos(u @ v / (np.linalg.norm(u) * np.linalg.norm(v))))

## Part A: Projections onto the axes (board, 10 Oct)

In [2]:
P1 = sp.Matrix([[1, 0], [0, 0]]); P2 = sp.Matrix([[0, 0], [0, 1]])
x = sp.Matrix([2, 3])
print("P_x1 (2,3) =", list(P1 * x), "| P_x1 again =", list(P1 * (P1 * x)))
print("P_x2 (2,3) =", list(P2 * x))
print("symmetric:", P1.T == P1, P2.T == P2, "| idempotent:", P1**2 == P1, P2**2 == P2)
print("P_x1 + P_x2 = I:", P1 + P2 == sp.eye(2), "| P_x1 P_x2 = 0:", P1 * P2 == sp.zeros(2))

P_x1 (2,3) = [2, 0] | P_x1 again = [2, 0]
P_x2 (2,3) = [0, 3]
symmetric: True True | idempotent: True True
P_x1 + P_x2 = I: True | P_x1 P_x2 = 0: True


## Part B: Dot product

In [3]:
u = np.array([1, 2, 3]); v = np.array([4, -5, 6]); w = np.array([2, 0, -1])
print("u.v =", u @ v, "| v.u =", v @ u, "| u^T v via matmul:", u.reshape(1, -1) @ v.reshape(-1, 1))
print("(u+v).w =", (u + v) @ w, "| u.w + v.w =", u @ w + v @ w)
print("(3u).w =", (3 * u) @ w, "| 3(u.w) =", 3 * (u @ w))
print("u.u =", u @ u, "| ||u|| =", np.linalg.norm(u).round(4))
print("(1,2).(-2,1) =", np.array([1, 2]) @ np.array([-2, 1]))

u.v = 12 | v.u = 12 | u^T v via matmul: [[12]]
(u+v).w = 1 | u.w + v.w = 1
(3u).w = -3 | 3(u.w) = -3
u.u = 14 | ||u|| = 3.7417
(1,2).(-2,1) = 0


## Part C: Angle, Cauchy–Schwarz, cosine similarity

In [4]:
u = np.array([2, 3]); v = np.array([4, 0])
c = u @ v / (np.linalg.norm(u) * np.linalg.norm(v))
print("cos theta =", round(c, 4), "= 2/sqrt(13) =", round(2 / np.sqrt(13), 4), "| theta =", round(angle_deg(u, v), 2), "deg")
u3 = np.array([1, 2, 2]); v3 = np.array([2, 1, 2])
print("3-D: u.v =", u3 @ v3, "| |u||v| =", np.linalg.norm(u3) * np.linalg.norm(v3), "| theta =", round(angle_deg(u3, v3), 2))
# Cauchy-Schwarz on random vectors
rng = np.random.default_rng(0)
ratios = []
for _ in range(10_000):
    a, b = rng.normal(size=5), rng.normal(size=5)
    ratios.append(abs(a @ b) / (np.linalg.norm(a) * np.linalg.norm(b)))
print("max |a.b|/(|a||b|) over 10,000 random pairs:", round(max(ratios), 4))
print("equality for parallel vectors:", (np.array([1, 2]) @ np.array([3, 6])), "=", np.linalg.norm([1, 2]) * np.linalg.norm([3, 6]))

cos theta = 0.5547 = 2/sqrt(13) = 0.5547 | theta = 56.31 deg
3-D: u.v = 8 | |u||v| = 9.0 | theta = 27.27
max |a.b|/(|a||b|) over 10,000 random pairs: 0.9836
equality for parallel vectors: 15 = 15.000000000000002


In [5]:
# cosine similarity of toy word-count vectors: [match, goal, election, vote]
docs = {"sports1": [3, 2, 0, 0], "sports2": [1, 4, 0, 0], "politics": [0, 0, 3, 4], "mixed": [1, 1, 1, 1]}
def cos(a, b):
    a, b = np.array(a, float), np.array(b, float)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
for a in docs:
    print(a.ljust(9), [round(float(cos(docs[a], docs[b])), 3) for b in docs])
print("scaling invariance: cos(sports1, 10*sports1) =", round(cos(docs["sports1"], 10 * np.array(docs["sports1"])), 3))

sports1   [1.0, 0.74, 0.0, 0.693]
sports2   [0.74, 1.0, 0.0, 0.606]
politics  [0.0, 0.0, 1.0, 0.7]
mixed     [0.693, 0.606, 0.7, 1.0]
scaling invariance: cos(sports1, 10*sports1) = 1.0


## Part D: Projection of a vector onto a vector

In [6]:
u = np.array([2, 3]); v = np.array([4, 0])
t = (u @ v) / (v @ v); p = t * v; e = u - p
print("t =", t, "| projection =", p, "| u - tv =", e, "| (u - tv).v =", e @ v)
u3 = np.array([1, 2, 2]); v3 = np.array([2, 1, 2])
t3 = sp.Rational(int(u3 @ v3), int(v3 @ v3)); p3 = t3 * sp.Matrix(v3); e3 = sp.Matrix(u3) - p3
print("3-D: t =", t3, "| proj =", list(p3), "| residual =", list(e3), "| residual.v =", (e3.T * sp.Matrix(v3))[0])
print("Pythagoras: |u|^2 =", u3 @ u3, "=", (p3.T * p3)[0] + (e3.T * e3)[0])

t = 0.5 | projection = [2. 0.] | u - tv = [0. 3.] | (u - tv).v = 0.0
3-D: t = 8/9 | proj = [16/9, 8/9, 16/9] | residual = [-7/9, 10/9, 2/9] | residual.v = 0
Pythagoras: |u|^2 = 9 = 9


## Part E: The projection matrix P = vvᵀ / vᵀv

In [7]:
for vec in ([1, 0], [1, 2], [2, 1, 2]):
    P = proj_matrix(vec)
    ev = sorted(P.eigenvals().items())
    print("v =", vec, "\nP =", P.tolist(), "\n  symmetric:", P.T == P, "| P^2 = P:", P * P == P,
          "| rank:", P.rank(), "| trace:", P.trace(), "| eigenvalues {value: multiplicity}:", dict(ev))
P = proj_matrix([1, 2])
print("P (3,1) =", list(P * sp.Matrix([3, 1])), "| P (1,2) =", list(P * sp.Matrix([1, 2])), "| P (-2,1) =", list(P * sp.Matrix([-2, 1])))
print("I - P =", (sp.eye(2) - P).tolist(), "| (I-P)^2 = I-P:", (sp.eye(2) - P)**2 == sp.eye(2) - P)
print("scaling v by 5 changes P?", proj_matrix([5, 10]) != P)

v = [1, 0] 
P = [[1, 0], [0, 0]] 
  symmetric: True | P^2 = P: True | rank: 1 | trace: 1 | eigenvalues {value: multiplicity}: {0: 1, 1: 1}
v = [1, 2] 
P = [[1/5, 2/5], [2/5, 4/5]] 
  symmetric: True | P^2 = P: True | rank: 1 | trace: 1 | eigenvalues {value: multiplicity}: {0: 1, 1: 1}
v = [2, 1, 2] 
P = [[4/9, 2/9, 4/9], [2/9, 1/9, 2/9], [4/9, 2/9, 4/9]] 
  symmetric: True | P^2 = P: True | rank: 1 | trace: 1 | eigenvalues {value: multiplicity}: {0: 2, 1: 1}
P (3,1) = [1, 2] | P (1,2) = [1, 2] | P (-2,1) = [0, 0]
I - P = [[4/5, -2/5], [-2/5, 1/5]] | (I-P)^2 = I-P: True
scaling v by 5 changes P? False


## Part F: Projection onto a subspace (column space)

In [8]:
A = sp.Matrix([[1, 0], [1, 1], [1, 2]]); b = sp.Matrix([6, 0, 0])
AtA = A.T * A; Atb = A.T * b
xhat = AtA.inv() * Atb
p = A * xhat; e = b - p
PA = A * AtA.inv() * A.T
print("A^T A =", AtA.tolist(), "| A^T b =", list(Atb), "| det =", AtA.det())
print("x_hat =", list(xhat), "| p =", list(p), "| e =", list(e), "| A^T e =", list(A.T * e))
print("P =", (6 * PA).tolist(), "/ 6 | symmetric:", PA.T == PA, "| idempotent:", PA * PA == PA, "| trace:", PA.trace())
print("|e|^2 =", (e.T * e)[0], "| eigenvalues:", PA.eigenvals())

A^T A = [[3, 3], [3, 5]] | A^T b = [6, 0] | det = 6
x_hat = [5, -3] | p = [5, 2, -1] | e = [1, -2, 1] | A^T e = [0, 0]
P = [[5, 2, -1], [2, 2, 2], [-1, 2, 5]] / 6 | symmetric: True | idempotent: True | trace: 2
|e|^2 = 6 | eigenvalues: {1: 2, 0: 1}


## Part G: Least squares by hand vs `lstsq` vs scikit-learn

In [9]:
x = np.array([1, 2, 3, 4], float); y = np.array([2, 3, 5, 6], float)
X = np.column_stack([np.ones_like(x), x])                     # beta = (c, m)
XtX = X.T @ X; Xty = X.T @ y
print("X^T X =", XtX.tolist(), "| X^T y =", Xty.tolist(), "| det =", round(np.linalg.det(XtX), 6))
beta = np.linalg.solve(XtX, Xty)
print("normal equations: beta (c, m) =", beta)
print("explicit (X^T X)^-1 X^T y   :", np.linalg.inv(XtX) @ X.T @ y)
print("np.linalg.lstsq             :", np.linalg.lstsq(X, y, rcond=None)[0])
print("np.linalg.pinv(X) @ y       :", np.linalg.pinv(X) @ y)
lr = LinearRegression().fit(x.reshape(-1, 1), y)
print("sklearn intercept, slope    :", round(lr.intercept_, 6), round(lr.coef_[0], 6))
yhat = X @ beta; e = y - yhat
print("fitted:", yhat, "| residuals:", e.round(4), "| X^T e:", (X.T @ e).round(10))
sse = e @ e; sst = ((y - y.mean())**2).sum()
print("SSE =", round(sse, 4), "| SST =", sst, "| R^2 =", round(1 - sse / sst, 4))
H = X @ np.linalg.inv(XtX) @ X.T
print("hat matrix H:\n", H, "\n trace(H) =", round(np.trace(H), 6), "| H^2 = H:", np.allclose(H @ H, H), "| H y = yhat:", np.allclose(H @ y, yhat))
print("leverages (diag H):", np.diag(H))
# column order (m, c) instead of (c, m)
X2 = np.column_stack([x, np.ones_like(x)])
print("columns [x, 1] -> beta =", np.linalg.solve(X2.T @ X2, X2.T @ y))

X^T X = [[4.0, 10.0], [10.0, 30.0]] | X^T y = [16.0, 47.0] | det = 20.0
normal equations: beta (c, m) = [0.5 1.4]
explicit (X^T X)^-1 X^T y   : [0.5 1.4]
np.linalg.lstsq             : [0.5 1.4]
np.linalg.pinv(X) @ y       : [0.5 1.4]
sklearn intercept, slope    : 0.5 1.4
fitted: [1.9 3.3 4.7 6.1] | residuals: [ 0.1 -0.3  0.3 -0.1] | X^T e: [0. 0.]
SSE = 0.2 | SST = 10.0 | R^2 = 0.98
hat matrix H:
 [[ 0.7  0.4  0.1 -0.2]
 [ 0.4  0.3  0.2  0.1]
 [ 0.1  0.2  0.3  0.4]
 [-0.2  0.1  0.4  0.7]] 
 trace(H) = 2.0 | H^2 = H: True | H y = yhat: True
leverages (diag H): [0.7 0.3 0.3 0.7]
columns [x, 1] -> beta = [1.4 0.5]


## Part H: Pseudo-inverse when columns are dependent

In [10]:
Xd = np.array([[1, 2], [1, 2], [1, 2]], float); yd = np.array([1, 2, 3], float)
G = Xd.T @ Xd
print("X^T X =", G.tolist(), "| det =", np.linalg.det(G), "| rank X =", np.linalg.matrix_rank(Xd))
try:
    np.linalg.inv(G)
    print("inv succeeded (numerically meaningless)")
except np.linalg.LinAlgError as err:
    print("np.linalg.inv:", err)
b_pinv = np.linalg.pinv(Xd) @ yd
print("pinv solution:", b_pinv, "| prediction:", Xd @ b_pinv, "| norm:", round(np.linalg.norm(b_pinv), 4))
for alt in ([2, 0], [0, 1], [1.2, 0.4]):
    alt = np.array(alt, float)
    print("alternative", alt, "-> prediction", Xd @ alt, "| norm", round(np.linalg.norm(alt), 4))
print("pinv(X) =\n", np.linalg.pinv(Xd))

X^T X = [[3.0, 6.0], [6.0, 12.0]] | det = 0.0 | rank X = 1
np.linalg.inv: Singular matrix
pinv solution: [0.4 0.8] | prediction: [2. 2. 2.] | norm: 0.8944
alternative [2. 0.] -> prediction [2. 2. 2.] | norm 2.0
alternative [0. 1.] -> prediction [2. 2. 2.] | norm 1.0
alternative [1.2 0.4] -> prediction [2. 2. 2.] | norm 1.2649
pinv(X) =
 [[0.0667 0.0667 0.0667]
 [0.1333 0.1333 0.1333]]


## Part I: Gram–Schmidt teaser

In [11]:
a1 = sp.Matrix([1, 1, 0]); a2 = sp.Matrix([1, 0, 1])
b2 = a2 - (a2.T * a1)[0] / (a1.T * a1)[0] * a1
print("a2 minus its projection on a1 =", list(b2), "| dot with a1 =", (b2.T * a1)[0])
q1 = a1 / a1.norm(); q2 = b2 / b2.norm()
Q = sp.Matrix.hstack(q1, q2)
print("q1 =", list(q1), "| q2 =", list(q2), "| Q^T Q =", sp.simplify(Q.T * Q).tolist())
print("projection matrix QQ^T equals A(A^TA)^-1A^T:", sp.simplify(Q * Q.T - sp.Matrix.hstack(a1, a2) * (sp.Matrix.hstack(a1, a2).T * sp.Matrix.hstack(a1, a2)).inv() * sp.Matrix.hstack(a1, a2).T) == sp.zeros(3))

a2 minus its projection on a1 = [1/2, -1/2, 1] | dot with a1 = 0
q1 = [sqrt(2)/2, sqrt(2)/2, 0] | q2 = [sqrt(6)/6, -sqrt(6)/6, sqrt(6)/3] | Q^T Q = [[1, 0], [0, 1]]
projection matrix QQ^T equals A(A^TA)^-1A^T: True


## Part J: Case studies

In [12]:
# J1: the mean is a projection onto the all-ones vector
y = np.array([2, 4, 9], float); one = np.ones(3)
print("projection of y onto 1 =", proj_vec(y, one), "| mean =", y.mean())
# J2: shadow of a 3-D point onto the ground plane z = 0 and onto the plane x + y - z = 0
n = sp.Matrix([1, 1, -1]); Pplane = sp.eye(3) - n * n.T / (n.T * n)[0]
print("P onto plane x+y-z=0:", (3 * Pplane).tolist(), "/3 | P (1,1,0) =", list(Pplane * sp.Matrix([1, 1, 0])))
# J3: recommendation by cosine similarity: user ratings for 4 movies
users = np.array([[5, 4, 0, 1], [4, 5, 1, 0], [0, 1, 5, 4]], float)
for i in range(3):
    print("user", i, [round(float(users[i] @ users[j] / np.linalg.norm(users[i]) / np.linalg.norm(users[j])), 3) for j in range(3)])

projection of y onto 1 = [5. 5. 5.] | mean = 5.0
P onto plane x+y-z=0: [[2, -1, 1], [-1, 2, 1], [1, 1, 2]] /3 | P (1,1,0) = [1/3, 1/3, 2/3]
user 0 [1.0, 0.952, 0.19]
user 1 [0.952, 1.0, 0.238]
user 2 [0.19, 0.238, 1.0]


## Part K: Practice-problem checks

In [13]:
print("P1:", np.array([2, -1, 3]) @ np.array([1, 4, 2]))
print("P2:", np.linalg.norm([3, 4, 12]))
print("P3:", np.array([1, 2, -1]) @ np.array([3, -1, 1]))
k = sp.symbols("k"); print("P4: k =", sp.solve(3 * k - 2 + 4, k))
print("P5:", round(angle_deg([1, 0], [1, 1]), 2), "| P6:", round(angle_deg([1, 2, 2], [2, -2, 1]), 2))
print("P7:", proj_vec([3, 4], [1, 0]), proj_vec([3, 4], [0, 1]))
print("P8:", proj_vec([4, 2], [1, 1]), "residual", np.array([4, 2]) - proj_vec([4, 2], [1, 1]))
P9 = proj_matrix([3, 4]); print("P9:", (25 * P9).tolist(), "/25 | P (5,0) =", list(P9 * sp.Matrix([5, 0])))
print("P10: P^2 = P:", P9 * P9 == P9, "| trace:", P9.trace())
M = sp.Matrix([[1, 1], [0, 0]]); print("P11: M^2 = M:", M * M == M, "| symmetric:", M == M.T, "| M (0,1) =", list(M * sp.Matrix([0, 1])))
print("P12:", proj_matrix([1, 1]).tolist())
print("P13: I - P for v=(1,2):", (5 * (sp.eye(2) - proj_matrix([1, 2]))).tolist(), "/5")
u = np.array([1, 2, 3]); v = np.array([4, 5, 6])
print("P15: u.v =", u @ v, "| |u||v| =", round(np.linalg.norm(u) * np.linalg.norm(v), 4), "| cos =", round(u @ v / np.linalg.norm(u) / np.linalg.norm(v), 4), "| angle =", round(angle_deg(u, v), 2))
d1 = np.array([2, 1, 0, 1]); d2 = np.array([1, 1, 1, 0])
print("P16:", d1 @ d2, round(d1 @ d2 / np.linalg.norm(d1) / np.linalg.norm(d2), 4))
X = sp.Matrix([[1, 0], [1, 1], [1, 2]]); yv = sp.Matrix([1, 3, 4])
bb = (X.T * X).inv() * X.T * yv; ee = yv - X * bb
print("P17: X^TX =", (X.T * X).tolist(), "X^Ty =", list(X.T * yv), "| beta =", list(bb), "| residuals =", list(ee), "| X^T e =", list(X.T * ee))
xs = np.array([1, 2, 3]); ys = np.array([2, 3, 7]); print("P18: m =", sp.Rational(int(xs @ ys), int(xs @ xs)), "=", round(xs @ ys / (xs @ xs), 4))
H = X * (X.T * X).inv() * X.T; print("P19: H =", (6 * H).tolist(), "/6 | trace =", H.trace())
A = sp.Matrix([[1, 0], [0, 1], [1, 1]]); PA = A * (A.T * A).inv() * A.T
print("P20: A^TA =", (A.T * A).tolist(), "| P =", (3 * PA).tolist(), "/3 | P (1,1,0) =", list(PA * sp.Matrix([1, 1, 0])))
n = sp.Matrix([1, 1, -1]); print("P21: I - nn^T/3 equals P20:", sp.eye(3) - n * n.T / 3 == PA)
Xd = np.array([[1, 2], [1, 2], [1, 2]], float); print("P22:", np.linalg.pinv(Xd) @ np.array([1, 2, 3], float))
a1 = sp.Matrix([1, 1, 1]); a2 = sp.Matrix([1, 0, 2]); print("P23:", list(a2 - (a2.T * a1)[0] / (a1.T * a1)[0] * a1))
print("P24:", P9.eigenvects())
print("P27:", proj_vec([2, 4, 9], [1, 1, 1]))

P1: 4
P2: 13.0
P3: 0
P4: k = [-2/3]
P5: 45.0 | P6: 90.0
P7: [3. 0.] [0. 4.]
P8: [3. 3.] residual [ 1. -1.]
P9: [[9, 12], [12, 16]] /25 | P (5,0) = [9/5, 12/5]
P10: P^2 = P: True | trace: 1
P11: M^2 = M: True | symmetric: False | M (0,1) = [1, 0]
P12: [[1/2, 1/2], [1/2, 1/2]]
P13: I - P for v=(1,2): [[4, -2], [-2, 1]] /5
P15: u.v = 32 | |u||v| = 32.8329 | cos = 0.9746 | angle = 12.93
P16: 3 0.7071
P17: X^TX = [[3, 3], [3, 5]] X^Ty = [8, 11] | beta = [7/6, 3/2] | residuals = [-1/6, 1/3, -1/6] | X^T e = [0, 0]
P18: m = 29/14 = 2.0714
P19: H = [[5, 2, -1], [2, 2, 2], [-1, 2, 5]] /6 | trace = 2
P20: A^TA = [[2, 1], [1, 2]] | P = [[2, -1, 1], [-1, 2, 1], [1, 1, 2]] /3 | P (1,1,0) = [1/3, 1/3, 2/3]
P21: I - nn^T/3 equals P20: True
P22: [0.4 0.8]
P23: [0, -1, 1]
P24: [(0, 1, [Matrix([
[-4/3],
[   1]])]), (1, 1, [Matrix([
[3/4],
[  1]])])]
P27: [5. 5. 5.]
